In [1]:
import pandas as pd 
import numpy as np

from pathlib import Path
import json

ROOT = Path.cwd()
DATA_PATH = ROOT.parent/"data" / "crop_production.csv"
OUT = ROOT / "outputs"

OUT.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Không thấy crop_production.csv trong: {ROOT}"
    )

df = pd.read_csv(DATA_PATH)

print("Initial shape:", df.shape)
display(df.head())

Initial shape: (246091, 7)


,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production
0,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Arecanut,1254.0,2000.0
1,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Other Kharif pulses,2.0,1.0
2,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Rice,102.0,321.0
3,Andaman and Nicobar Islands,NICOBARS,2000,Whole Year,Banana,176.0,641.0
4,Andaman and Nicobar Islands,NICOBARS,2000,Whole Year,Cashewnut,720.0,165.0


In [2]:
# Check dataset structure, missing values, duplicates,
# and basic invalid values
print("DATA INFO")
df.info()
print("\nMISSING VALUES")
print(df.isnull().sum())
print("\nDUPLICATES ")
print(df.duplicated().sum())
print("\nINVALID VALUES")
print("Area <= 0:", (df["Area"] <= 0).sum())
print("Production < 0:", (df["Production"] < 0).sum())

DATA INFO
<class 'pandas.DataFrame'>
RangeIndex: 246091 entries, 0 to 246090
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   State_Name     246091 non-null  str    
 1   District_Name  246091 non-null  str    
 2   Crop_Year      246091 non-null  int64  
 3   Season         246091 non-null  str    
 4   Crop           246091 non-null  str    
 5   Area           246091 non-null  float64
 6   Production     242361 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 13.1 MB

MISSING VALUES
State_Name          0
District_Name       0
Crop_Year           0
Season              0
Crop                0
Area                0
Production       3730
dtype: int64

DUPLICATES 
0

INVALID VALUES
Area <= 0: 0
Production < 0: 0


In [3]:
# Remove completely duplicated rows
df = df.drop_duplicates().copy()

# Remove rows with missing values in required variables
required_cols = [
    "State_Name",
    "District_Name",
    "Crop_Year",
    "Season",
    "Crop",
    "Area",
    "Production"
]
df = df.dropna(subset=required_cols).copy()
# Area must be greater than 0 because Yield = Production / Area
df = df[df["Area"] > 0].copy()
# Production cannot be negative
df = df[df["Production"] >= 0].copy()
print("Shape after basic cleaning:", df.shape)

Shape after basic cleaning: (242361, 7)


In [4]:
# Remove unnecessary leading and trailing spaces
# from categorical/text variables
text_cols = [
    "State_Name",
    "District_Name",
    "Season",
    "Crop"
]

for col in text_cols:
    df[col] = df[col].str.strip()
# Ensure numerical variables have the correct data type
df["Crop_Year"] = pd.to_numeric(df["Crop_Year"], errors="coerce")
df["Area"] = pd.to_numeric(df["Area"], errors="coerce")
df["Production"] = pd.to_numeric(df["Production"], errors="coerce")

# Remove rows with invalid numeric conversions
df = df.dropna(
    subset=["Crop_Year", "Area", "Production"]
).copy()
print("Shape after standardization:", df.shape)

Shape after standardization: (242361, 7)


In [5]:
# Create a new variable "District_ID" by combining "State_Name" and "District_Name"
df["District_ID"] = (
    df["State_Name"] + " / " + df["District_Name"]
)

In [6]:
# Calculate Yield as Production divided by Area
df["Yield"] = df["Production"] / df["Area"]
# Check the validity of the calculated Yield
print("Missing Yield:", df["Yield"].isnull().sum())
print("Infinite Yield:", np.isinf(df["Yield"]).sum())
print("Negative Yield:", (df["Yield"] < 0).sum())
display(df[["Area", "Production", "Yield"]].head())

Missing Yield: 0
Infinite Yield: 0
Negative Yield: 0


,Area,Production,Yield
0,1254.0,2000.0,1.594896
1,2.0,1.0,0.500000
2,102.0,321.0,3.147059
3,176.0,641.0,3.642045
4,720.0,165.0,0.229167


In [7]:
# Detect Yield outliers using the IQR method for each Crop
reference = df[
    df["Crop_Year"].between(1998, 2010)
].copy()

# So sánh năng suất trong cùng bang, cây và mùa.
grouped_yield = reference.groupby(
    ["State_Name", "Crop", "Season"]
)["Yield"]

# Tính Q1, Q3 và số quan sát hợp lệ của mỗi nhóm.
q1 = grouped_yield.quantile(0.25).rename("Q1")
q3 = grouped_yield.quantile(0.75).rename("Q3")
reference_n = grouped_yield.count().rename("Reference_N")

# Gộp kết quả thành bảng ngưỡng.
limits = pd.concat(
    [q1, q3, reference_n],
    axis=1
).reset_index()

limits["IQR"] = limits["Q3"] - limits["Q1"]

limits["Lower"] = limits["Q1"] - 1.5 * limits["IQR"]
limits["Upper"] = limits["Q3"] + 1.5 * limits["IQR"]

# Gắn ngưỡng tương ứng vào từng dòng dữ liệu.
df = df.merge(
    limits[
        [
            "State_Name", "Crop", "Season",
            "Reference_N", "Lower", "Upper"
        ]
    ],
    on=["State_Name", "Crop", "Season"],
    how="left",
    validate="many_to_one"
)

# Chỉ đánh giá nhóm có ít nhất 10 quan sát tham chiếu
# và có đủ hai ngưỡng.
known = (
    (df["Reference_N"] >= 10)
    & df["Lower"].notna()
    & df["Upper"].notna()
)

# Ban đầu để NA: chưa đủ thông tin để đánh giá.
df["Outlier_Flag"] = pd.Series(
    pd.NA,
    index=df.index,
    dtype="boolean"
)

# Với những dòng đủ thông tin:
# True nếu nằm ngoài ngưỡng, False nếu nằm trong ngưỡng.
df.loc[known, "Outlier_Flag"] = (
    (df.loc[known, "Yield"] < df.loc[known, "Lower"])
    | (df.loc[known, "Yield"] > df.loc[known, "Upper"])
)

# Thống kê kết quả.
outlier_count = df["Outlier_Flag"].sum()
evaluated_count = df["Outlier_Flag"].notna().sum()
unknown_count = df["Outlier_Flag"].isna().sum()

print("Số dòng được đánh giá:", evaluated_count)
print("Số dòng được gắn cờ ngoại lệ:", outlier_count)
print("Số dòng chưa đủ dữ liệu tham chiếu:", unknown_count)

if evaluated_count > 0:
    print(
        "Tỷ lệ ngoại lệ trong các dòng đã được đánh giá:",
        round(outlier_count / evaluated_count * 100, 2),
        "%"
    )

# Xóa cột phụ, giữ Outlier_Flag.
df = df.drop(
    columns=["Reference_N", "Lower", "Upper"]
)

display(
    df[
        [
            "State_Name",
            "District_Name",
            "Crop_Year",
            "Crop",
            "Season",
            "Yield",
            "Outlier_Flag"
        ]
    ].head(10)
)

Số dòng được đánh giá: 235455
Số dòng được gắn cờ ngoại lệ: 14281
Số dòng chưa đủ dữ liệu tham chiếu: 6906
Tỷ lệ ngoại lệ trong các dòng đã được đánh giá: 6.07 %


,State_Name,District_Name,Crop_Year,Crop,Season,Yield,Outlier_Flag
0,Andaman and Nicobar Islands,NICOBARS,2000,Arecanut,Kharif,1.594896,<NA>
1,Andaman and Nicobar Islands,NICOBARS,2000,Other Kharif pulses,Kharif,0.500000,<NA>
2,Andaman and Nicobar Islands,NICOBARS,2000,Rice,Kharif,3.147059,False
3,Andaman and Nicobar Islands,NICOBARS,2000,Banana,Whole Year,3.642045,False
4,Andaman and Nicobar Islands,NICOBARS,2000,Cashewnut,Whole Year,0.229167,False
5,Andaman and Nicobar Islands,NICOBARS,2000,Coconut,Whole Year,3583.223250,False
6,Andaman and Nicobar Islands,NICOBARS,2000,Dry ginger,Whole Year,2.777778,False
7,Andaman and Nicobar Islands,NICOBARS,2000,Sugarcane,Whole Year,2.000000,False
8,Andaman and Nicobar Islands,NICOBARS,2000,Sweet potato,Whole Year,3.000000,<NA>
9,Andaman and Nicobar Islands,NICOBARS,2000,Tapioca,Whole Year,4.225000,<NA>


In [8]:
# Outliers are retained because the IQR method only identifies
# observations that are far from the main distribution.
# It does not prove that these observations are data errors.
# Therefore, no outliers are removed from the cleaned dataset.
print("Outliers are detected but retained in the dataset.")

Outliers are detected but retained in the dataset.


In [9]:
# Perform a final quality check before handing the dataset
# to the other team members
print("FINAL QUALITY CHECK")
print("Shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Area <= 0:", (df["Area"] <= 0).sum())
print("Production < 0:", (df["Production"] < 0).sum())
print(
    "Invalid Yield:",
    df["Yield"].isnull().sum()
    + np.isinf(df["Yield"]).sum()
    + (df["Yield"] < 0).sum()
)

FINAL QUALITY CHECK
Shape: (242361, 10)
Missing values: 6906
Duplicate rows: 0
Area <= 0: 0
Production < 0: 0
Invalid Yield: 0


In [10]:
# Overall descriptive statistics for Yield
overall_stats = df["Yield"].agg([
    "count",
    "mean",
    "median",
    "min",
    "max",
    "std",
    "var"
])
print("OVERALL YIELD STATISTICS")
display(overall_stats)

# Descriptive statistics by Crop
crop_stats = df.groupby("Crop")["Yield"].agg(
    Count="count",
    Mean="mean",
    Median="median",
    Min="min",
    Max="max",
    Std="std",
    Variance="var"
).reset_index()

print("\nSTATISTICS BY CROP")
display(crop_stats)

# Descriptive statistics by Season
season_stats = df.groupby("Season")["Yield"].agg(
    Count="count",
    Mean="mean",
    Median="median",
    Min="min",
    Max="max",
    Std="std",
    Variance="var"
).reset_index()

print("\nSTATISTICS BY SEASON")
display(season_stats)

OVERALL YIELD STATISTICS


count     242361.000000
mean          41.649059
median         1.000000
min            0.000000
max        88000.000000
std          817.572839
var       668425.347112
Name: Yield, dtype: float64


STATISTICS BY CROP


,Crop,Count,Mean,Median,Min,Max,Std,Variance
0,Apple,4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,Arcanut (Processed),20,1.284838,1.397341,0.709677,1.783976,0.295224,0.087157
2,Arecanut,1538,2.163932,1.198270,0.039936,19.300000,2.631990,6.927373
3,Arhar/Tur,7476,0.797443,0.731169,0.000000,8.000000,0.433312,0.187759
4,Ash Gourd,44,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...
119,Wheat,7878,2.092279,1.890783,0.000000,7.461538,1.058784,1.121023
120,Yam,36,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
121,other fibres,10,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
122,other misc. pulses,70,0.243393,0.166151,0.000000,1.028346,0.250030,0.062515



STATISTICS BY SEASON


,Season,Count,Mean,Median,Min,Max,Std,Variance
0,Autumn,4930,2.722770,1.002784,0.017750,1494.00000,41.074750,1.687135e+03
1,Kharif,94283,2.300026,0.815287,0.000000,370.00000,7.383673,5.451863e+01
2,Rabi,66160,2.032057,0.909882,0.000000,108.44029,4.003536,1.602830e+01
3,Summer,14811,2.810451,1.557468,0.000000,235.00000,5.752926,3.309615e+01
4,Whole Year,56127,171.944906,3.000000,0.000000,88000.00000,1692.325429,2.863965e+06
5,Winter,6050,6.117808,1.220235,0.015994,111.20000,15.755087,2.482228e+02


In [11]:
# Select the 12 crops with the most observations
top_12_crops = df["Crop"].value_counts().head(12).index

df_model = df[df["Crop"].isin(top_12_crops)].copy()

print("SELECTED CROPS FOR MODELING")
print(top_12_crops.tolist())

print("\nNumber of crops:", df_model["Crop"].nunique())
print("Modeling dataset shape:", df_model.shape)

print("\nSAMPLES PER CROP")
display(df_model["Crop"].value_counts())

SELECTED CROPS FOR MODELING
['Rice', 'Maize', 'Moong(Green Gram)', 'Urad', 'Sesamum', 'Groundnut', 'Wheat', 'Sugarcane', 'Rapeseed &Mustard', 'Arhar/Tur', 'Gram', 'Jowar']

Number of crops: 12
Modeling dataset shape: (111207, 10)

SAMPLES PER CROP


Crop
Rice                 15082
Maize                13787
Moong(Green Gram)    10106
Urad                  9710
Sesamum               8821
Groundnut             8770
Wheat                 7878
Sugarcane             7827
Rapeseed &Mustard     7533
Arhar/Tur             7476
Gram                  7227
Jowar                 6990
Name: count, dtype: int64

In [14]:
# Create a simple Data Dictionary
data_dictionary = pd.DataFrame({
    "Variable": [
        "State_Name",
        "District_Name",
        "Crop_Year",
        "Season",
        "Crop",
        "Area",
        "Production",
        "Yield"
    ],

    "Description": [
        "Name of the state or region",
        "Name of the district or region",
        "Year of crop production",
        "Cropping season",
        "Type of crop",
        "Cultivated area",
        "Crop production",
        "Yield calculated as Production / Area"
    ],

    "Type": [
        "Categorical",
        "Categorical",
        "Integer",
        "Categorical",
        "Categorical",
        "Numeric",
        "Numeric",
        "Numeric"
    ]
})

print("DATA DICTIONARY")
display(data_dictionary)
# Export the cleaned dataset for TV2 and TV3
df.to_csv(OUT / "cleaned_data.csv", index=False)

print("\nCleaned data exported successfully:")
print("cleaned_data.csv")

DATA DICTIONARY


,Variable,Description,Type
0,State_Name,Name of the state or region,Categorical
1,District_Name,Name of the district or region,Categorical
2,Crop_Year,Year of crop production,Integer
3,Season,Cropping season,Categorical
4,Crop,Type of crop,Categorical
5,Area,Cultivated area,Numeric
6,Production,Crop production,Numeric
7,Yield,Yield calculated as Production / Area,Numeric



Cleaned data exported successfully:
cleaned_data.csv


In [13]:
# Export modeling dataset
df_model.to_csv(OUT / "model_data_top12.csv", index=False)

config = {
    "states": df_model["State_Name"].unique().tolist(),
    "crops": df_model["Crop"].unique().tolist(),
    "seasons": df_model["Season"].unique().tolist(),
    "features": df_model.columns.tolist(),
}
(OUT / "config.json").write_text(
    json.dumps(config, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("File exported successfully: model_data_top12.csv")

File exported successfully: model_data_top12.csv
